# Results figures

Parses the raw experiment rows under `artifacts/paper/raw/` and
regenerates every evaluation figure in the paper. The raw `.jsonl`
files come from the Zenodo archive described in `ARTIFACT.md`.

In [ ]:
import json
import math
from collections import defaultdict
from pathlib import Path

import lusca  # noqa: F401  (registers the "lusca" matplotlib style)
import matplotlib.pyplot as plt
import numpy as np

%config InlineBackend.figure_formats = ['retina']

ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "pyproject.toml").exists()
)
RAW = ROOT / "artifacts/paper/raw"
FIG_DIR = ROOT / "build/figures"


def save_figure(figure, name):
    """Write ``figure`` as ``build/figures/<name>.pdf``, byte-stable across runs."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    figure.savefig(FIG_DIR / f"{name}.pdf", metadata={"CreationDate": None})


TOPOLOGIES = ("grid", "hex_lattice")
LOGICAL, GREEDY, PRISM = "logical", "greedy", "prism"
BENCHMARKS = (
    "dj",
    "qaoa",
    "draper_qft_adder",
    "modular_adder",
    "qft",
    "qpeexact",
    "qpeinexact",
    "qftentangled",
    "bmw_quark_copula",
    "hhl",
    "ae",
    "cdkm_ripple_carry_adder",
    "vbe_ripple_carry_adder",
    "vqe_two_local",
    "rg_qft_multiplier",
    "multiplier",
)


def load_rows(path, expected_rows):
    """Parse a raw jsonl file, requiring the exact row count and all-success."""
    text = path.read_text()
    rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    if len(rows) != expected_rows or any(not row.get("success") for row in rows):
        raise RuntimeError(f"invalid result population: {path}")
    return rows


def percent_range(rows, field):
    """Format the min-max of a fractional field as a percent range."""
    values = sorted(100.0 * float(row[field]) for row in rows)
    return f"{values[0]:.1f}-{values[-1]:.1f}%"

## Core comparison

Regenerates the paper's two core evaluation figures from the validated
20-seed runs: the aggregate comparison of all three routing arms, and the
per-benchmark PRISM-vs-Greedy reduction panels split by topology.

In [ ]:
CONTROLLED_RAW = RAW / "core.jsonl"
LOGICAL_RAW = RAW / "logical.jsonl"

SEEDS = tuple(range(0, 2000, 100))
ARMS = (LOGICAL, GREEDY, PRISM)
METRICS = ("router_inserted_swaps", "physical_depth")

TOPOLOGY_LABELS = {
    "grid": "Square grid",
    "hex_lattice": "Hexagonal lattice",
}
ARM_LABELS = {
    LOGICAL: "Logical-SABRE",
    GREEDY: "Greedy",
    PRISM: "PRISM",
}
ARM_COLORS = {
    LOGICAL: "#b8bdc2",
    GREEDY: "#d69b32",
    PRISM: "#39877f",
}
ARM_HATCHES = {
    LOGICAL: "///",
    GREEDY: "...",
    PRISM: "",
}
BENCHMARK_LABELS = {
    "dj": "dj",
    "qaoa": "qaoa",
    "draper_qft_adder": "draper",
    "modular_adder": "mod-add",
    "qft": "qft",
    "qpeexact": "qpe-ex",
    "qpeinexact": "qpe-in",
    "qftentangled": "qft-ent",
    "bmw_quark_copula": "copula",
    "hhl": "hhl",
    "ae": "ae",
    "cdkm_ripple_carry_adder": "cdkm",
    "vbe_ripple_carry_adder": "vbe",
    "vqe_two_local": "vqe-2local",
    "rg_qft_multiplier": "rg-mult",
    "multiplier": "mult",
}
REGRESSION = "#b55a55"

### Load validated raw rows

We require the expected row count (1280 controlled rows plus 640
logical-router rows), every row a successful compile, and the combined
population covering the full benchmark x topology x seed x arm grid exactly
once.

In [ ]:
core_rows = load_rows(CONTROLLED_RAW, 1280)
core_rows += load_rows(LOGICAL_RAW, 640)
expected = {
    (benchmark, topology, seed, arm)
    for benchmark in BENCHMARKS
    for topology in TOPOLOGIES
    for seed in SEEDS
    for arm in ARMS
}
observed = {
    (
        row["circuit_name"],
        row["topology_name"],
        int(row["outer_seed"]),
        row["strategy"],
    )
    for row in core_rows
}
if observed != expected or len(core_rows) != len(expected):
    raise RuntimeError("the combined core population is incomplete")
print(f"{len(core_rows)} validated rows covering the full arm grid")

### Controlled comparison

Average each metric over the 20 seeds per (benchmark, topology, arm) cell,
then derive three tables: suite-level absolute means per arm, per-benchmark
means with the PRISM-vs-Greedy relative reduction, and the controlled
comparison with suite-total and equal-workload reductions plus
win/tie/loss counts.

In [ ]:
def _summaries(rows):
    values = defaultdict(list)
    for row in rows:
        for metric in METRICS:
            values[
                (
                    row["circuit_name"],
                    row["topology_name"],
                    row["strategy"],
                    metric,
                )
            ].append(float(row[metric]))

    per_benchmark = []
    absolute = []
    comparisons = []
    for metric in METRICS:
        benchmark_means = {}
        for benchmark in BENCHMARKS:
            for topology in TOPOLOGIES:
                for arm in ARMS:
                    cell = values[(benchmark, topology, arm, metric)]
                    if len(cell) != len(SEEDS):
                        raise RuntimeError(
                            f"incomplete seed cell: {benchmark}, {topology}, "
                            f"{arm}, {metric}"
                        )
                    benchmark_means[(benchmark, topology, arm)] = float(np.mean(cell))

        for topology in TOPOLOGIES:
            for arm in ARMS:
                absolute.append(
                    {
                        "topology": topology,
                        "metric": metric,
                        "strategy": arm,
                        "mean": float(
                            np.mean(
                                [
                                    benchmark_means[(benchmark, topology, arm)]
                                    for benchmark in BENCHMARKS
                                ]
                            )
                        ),
                    }
                )

            greedy = np.asarray(
                [
                    benchmark_means[(benchmark, topology, GREEDY)]
                    for benchmark in BENCHMARKS
                ]
            )
            prism = np.asarray(
                [
                    benchmark_means[(benchmark, topology, PRISM)]
                    for benchmark in BENCHMARKS
                ]
            )
            defined = greedy > 0
            reductions = np.full_like(greedy, np.nan)
            reductions[defined] = 1.0 - prism[defined] / greedy[defined]
            zero_zero = (~defined) & (prism == 0)
            comparisons.append(
                {
                    "topology": topology,
                    "metric": metric,
                    "greedy_mean": float(np.mean(greedy)),
                    "prism_mean": float(np.mean(prism)),
                    "suite_total_reduction": float(
                        1.0 - np.sum(prism) / np.sum(greedy)
                    ),
                    "equal_workload_reduction": float(np.mean(reductions[defined])),
                    "wins": int(np.sum(reductions[defined] > 1e-12)),
                    "ties": int(
                        np.sum(np.abs(reductions[defined]) <= 1e-12) + np.sum(zero_zero)
                    ),
                    "losses": int(np.sum(reductions[defined] < -1e-12)),
                    "undefined_nonzero": int(np.sum((~defined) & (prism != 0))),
                }
            )

            for index, benchmark in enumerate(BENCHMARKS):
                per_benchmark.append(
                    {
                        "benchmark": benchmark,
                        "topology": topology,
                        "metric": metric,
                        "logical_sabre_mean": benchmark_means[
                            (benchmark, topology, LOGICAL)
                        ],
                        "greedy_mean": float(greedy[index]),
                        "prism_mean": float(prism[index]),
                        "relative_reduction": (
                            float(reductions[index]) if defined[index] else None
                        ),
                    }
                )
    return absolute, per_benchmark, comparisons


absolute, per_benchmark, comparisons = _summaries(core_rows)

header = (
    f"{'topology':<12} {'metric':<22} {'greedy':>8} {'prism':>8} "
    f"{'suite':>7} {'equal':>7}  W/T/L"
)
print(header)
for row in comparisons:
    print(
        f"{row['topology']:<12} {row['metric']:<22} "
        f"{row['greedy_mean']:>8.2f} {row['prism_mean']:>8.2f} "
        f"{100 * row['suite_total_reduction']:>6.1f}% "
        f"{100 * row['equal_workload_reduction']:>6.1f}%  "
        f"{row['wins']}/{row['ties']}/{row['losses']}"
    )

### Aggregate comparison

Suite-mean routing SWAPs and physical depth per arm, grouped by topology.

In [ ]:
lookup = {
    (row["topology"], row["metric"], row["strategy"]): row["mean"] for row in absolute
}
with plt.style.context("lusca"):
    plt.rcParams.update(
        {
            "font.size": 7.8,
            "axes.labelsize": 8.2,
            "xtick.labelsize": 7.7,
            "ytick.labelsize": 7.4,
            "legend.fontsize": 7.8,
            "pdf.fonttype": 42,
        }
    )
    figure, axes = plt.subplots(1, 2, figsize=(7.15, 1.65))
    group_x = np.arange(len(TOPOLOGIES))
    width = 0.22
    specs = (
        ("router_inserted_swaps", "(a) Routing SWAPs"),
        ("physical_depth", "(b) Physical depth"),
    )
    for axis, (metric, title) in zip(axes, specs, strict=True):
        all_values = []
        for arm_index, arm in enumerate(ARMS):
            arm_values = [lookup[(topology, metric, arm)] for topology in TOPOLOGIES]
            all_values.extend(arm_values)
            bars = axis.bar(
                group_x + (arm_index - 1) * width,
                arm_values,
                width=width,
                color=ARM_COLORS[arm],
                edgecolor="#303030",
                linewidth=0.5,
                label=ARM_LABELS[arm],
                zorder=3,
            )
            for bar in bars:
                bar.set_hatch(ARM_HATCHES[arm])
            axis.bar_label(bars, fmt="%.1f", padding=2, fontsize=7.0)
        axis.set_title(title, loc="left", fontsize=8.2)
        axis.set_xticks(
            group_x,
            [TOPOLOGY_LABELS[topology] for topology in TOPOLOGIES],
        )
        axis.set_ylim(0, max(all_values) * 1.19)
        axis.grid(axis="y", color="#dedede", linewidth=0.5, zorder=0)
        axis.set_axisbelow(True)
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.01),
    )
    figure.tight_layout(
        rect=(0, 0, 1, 0.82),
        pad=0.3,
        w_pad=0.8,
    )
save_figure(figure, "eval_core_aggregate_row_v22")
plt.show()

### Per-benchmark reductions by topology

PRISM's percentage reduction versus Greedy for every benchmark, one panel
per (topology, metric). Benchmarks are ordered by their reduction in
routing SWAPs pooled across both topologies; red bars mark regressions.

In [ ]:
lookup = {
    (row["benchmark"], row["topology"], row["metric"]): row for row in per_benchmark
}
pooled = defaultdict(lambda: [0.0, 0.0])
for row in per_benchmark:
    key = (row["benchmark"], row["metric"])
    pooled[key][0] += float(row["greedy_mean"])
    pooled[key][1] += float(row["prism_mean"])


def pooled_reduction(benchmark, metric):
    greedy, prism = pooled[(benchmark, metric)]
    if greedy == 0:
        if prism == 0:
            return 0.0
        raise RuntimeError(f"undefined pooled reduction: {benchmark}, {metric}")
    return 1.0 - prism / greedy


order = sorted(
    BENCHMARKS,
    key=lambda benchmark: (
        pooled_reduction(benchmark, "router_inserted_swaps"),
        benchmark,
    ),
)
panel_specs = (
    ("grid", "router_inserted_swaps", "(a) Square grid \u2014 routing SWAPs"),
    ("grid", "physical_depth", "(b) Square grid \u2014 physical depth"),
    (
        "hex_lattice",
        "router_inserted_swaps",
        "(c) Hexagonal lattice \u2014 routing SWAPs",
    ),
    (
        "hex_lattice",
        "physical_depth",
        "(d) Hexagonal lattice \u2014 physical depth",
    ),
)
reductions = {}
for topology, metric, _title in panel_specs:
    values = []
    for benchmark in order:
        row = lookup[(benchmark, topology, metric)]
        greedy = float(row["greedy_mean"])
        prism = float(row["prism_mean"])
        if greedy == 0:
            if prism != 0:
                raise RuntimeError(
                    f"undefined reduction: {benchmark}, {topology}, {metric}"
                )
            values.append(0.0)
        else:
            values.append(100.0 * (1.0 - prism / greedy))
    reductions[(topology, metric)] = np.asarray(values)

with plt.style.context("lusca"):
    plt.rcParams.update(
        {
            "font.size": 7.2,
            "axes.labelsize": 7.6,
            "xtick.labelsize": 6.2,
            "ytick.labelsize": 6.8,
            "pdf.fonttype": 42,
        }
    )
    figure, axes = plt.subplots(
        2,
        2,
        figsize=(7.15, 3.18),
        sharex=True,
    )
    x = np.arange(len(order))
    for axis, (topology, metric, title) in zip(
        axes.flat,
        panel_specs,
        strict=True,
    ):
        values = reductions[(topology, metric)]
        colors = np.where(
            values >= 0,
            ARM_COLORS[PRISM],
            REGRESSION,
        )
        axis.bar(
            x,
            values,
            color=colors,
            edgecolor="#303030",
            linewidth=0.35,
            width=0.76,
            zorder=3,
        )
        axis.axhline(0, color="#777777", linewidth=0.6, zorder=2)
        axis.set_title(title, loc="left", fontsize=7.8)
        axis.set_ylabel("PRISM reduction\nvs. Greedy (%)")
        axis.grid(axis="y", color="#e1e1e1", linewidth=0.45, zorder=0)
        axis.set_axisbelow(True)

    for column, metric in enumerate(METRICS):
        all_values = np.concatenate(
            [reductions[(topology, metric)] for topology in TOPOLOGIES]
        )
        upper = max(0.0, float(np.max(all_values)))
        upper_margin = max(2.0, 0.08 * upper)
        if metric == "router_inserted_swaps":
            for row in range(2):
                axes[row, column].set_ylim(0.0, upper + upper_margin)
            continue

        axes[0, column].set_ylim(0.0, upper + upper_margin)
        lower = min(
            0.0,
            float(np.min(reductions[("hex_lattice", "physical_depth")])),
        )
        lower_margin = max(1.0, 0.08 * (upper - lower))
        axes[1, column].set_ylim(
            lower - lower_margin,
            upper + upper_margin,
        )

    labels = [BENCHMARK_LABELS[benchmark] for benchmark in order]
    for axis in axes[0]:
        axis.tick_params(axis="x", labelbottom=False)
    for axis in axes[1]:
        axis.set_xticks(x, labels, rotation=52, ha="right")
    figure.tight_layout(pad=0.35, h_pad=0.7, w_pad=0.8)
save_figure(figure, "eval_core_by_benchmark_topologies_v22")
plt.show()

### Headline ranges

Recompute the two headline ranges quoted in the paper: the suite-total
reduction across the two topologies for routing SWAPs and for physical
depth.

In [ ]:
EXPECTED_SWAP_RANGE = "26.0-31.3%"
EXPECTED_DEPTH_RANGE = "7.9-12.4%"

swap_range = percent_range(
    [row for row in comparisons if row["metric"] == "router_inserted_swaps"],
    "suite_total_reduction",
)
depth_range = percent_range(
    [row for row in comparisons if row["metric"] == "physical_depth"],
    "suite_total_reduction",
)
print(f"Suite-total SWAP reduction range: {swap_range}")
print(f"Suite-total depth reduction range: {depth_range}")
assert swap_range == EXPECTED_SWAP_RANGE, swap_range
assert depth_range == EXPECTED_DEPTH_RANGE, depth_range

## Scaling

Regenerates the paper's combined scaling figure from the scale width-sweep
experiment: 16 benchmarks x 7 widths (8 to 32 logical qubits) x 2
topologies x 5 seeds x 2 routing arms (greedy early-commit vs. PRISM
retained-set), 2240 runs total.

Panels (a) and (b) show suite-level reductions in routing SWAPs and
critical-path duration; panel (c) shows mean compilation time per arm
and topology.

In [ ]:
SCALE_RAW = RAW / "scale.jsonl"

WIDTHS = (8, 12, 16, 20, 24, 28, 32)
ARMS = (GREEDY, PRISM)
SEEDS = (0, 100, 200, 300, 400)

GRID_COLOR = "#4c78a8"
HEX_COLOR = "#d9902f"
GREEDY_COLOR = "#d69b32"
PRISM_COLOR = "#39877f"

### Load and validate

The row population must
cover the full benchmark x width x topology x seed x arm grid exactly
once, and every row must carry the frozen experiment configuration
(SABRE heuristic, load tie-break, 43 starts, 16
iterations, 40 final swap trials). Any drift aborts the notebook.

In [ ]:
scale_rows = load_rows(SCALE_RAW, 2240)
expected = {
    (benchmark, width, topology, seed, arm)
    for benchmark in BENCHMARKS
    for width in WIDTHS
    for topology in TOPOLOGIES
    for seed in SEEDS
    for arm in ARMS
}
observed = {
    (
        row["circuit_name"],
        int(row["requested_n"]),
        row["topology_name"],
        int(row["outer_seed"]),
        row["strategy"],
    )
    for row in scale_rows
}
if len(scale_rows) != len(expected) or observed != expected:
    raise RuntimeError("scaling population drift")

for row in scale_rows:
    trace = row.get("search_trace") or {}
    duration = float(row["duration_ns"])
    if (
        not row.get("success")
        or row.get("schema_version") != 5
        or row.get("objective") != "swaps"
        or row.get("heuristic_config") != "qiskit_sabre"
        or row.get("tie_break") != "load"
        or row.get("duration_unmodeled_ops")
        or row.get("runner_workers") != 12
        or trace.get("n_starts") != 43
        or trace.get("max_iterations") != 16
        or trace.get("final_swap_trials") != 40
        or trace.get("n_failed_starts") != 0
        or not math.isclose(duration / 100.0, round(duration / 100.0))
    ):
        raise RuntimeError(f"invalid scaling row: {row.get('run_id')}")

print(f"{len(scale_rows)} rows validated")

### Summaries

Average the five seeds within each workload (benchmark, width,
topology, arm). Scaling reductions use the ratio of the sums of the 16
workload means; the table also reports the unweighted mean of
workload-relative reductions plus win/tie/loss counts. Timing points
average the 16 workload means of per-compilation wall time.

In [ ]:
samples = defaultdict(list)
for row in scale_rows:
    prefix = (
        row["circuit_name"],
        int(row["requested_n"]),
        row["topology_name"],
        row["strategy"],
    )
    for metric in (
        "router_inserted_swaps",
        "physical_depth",
        "duration_ns",
        "compile_time_s",
    ):
        samples[(*prefix, metric)].append(float(row[metric]))

scaling = []
timing = []
for width in WIDTHS:
    for topology in TOPOLOGIES:
        for metric in (
            "router_inserted_swaps",
            "physical_depth",
            "duration_ns",
        ):
            values = {arm: [] for arm in ARMS}
            for benchmark in BENCHMARKS:
                for arm in ARMS:
                    group = samples[(benchmark, width, topology, arm, metric)]
                    if len(group) != len(SEEDS):
                        raise RuntimeError(
                            f"incomplete metric cell: {benchmark}, "
                            f"{width}, {topology}, {arm}, {metric}"
                        )
                    values[arm].append(float(np.mean(group)))
            greedy = values[GREEDY]
            prism = values[PRISM]
            relative = [
                1.0 - retained / fixed
                for fixed, retained in zip(greedy, prism, strict=True)
                if fixed != 0
            ]
            wins = sum(
                retained < fixed for fixed, retained in zip(greedy, prism, strict=True)
            )
            ties = sum(
                math.isclose(
                    retained,
                    fixed,
                    rel_tol=0.0,
                    abs_tol=1e-12,
                )
                for fixed, retained in zip(greedy, prism, strict=True)
            )
            scaling.append(
                {
                    "width": width,
                    "topology": topology,
                    "metric": metric,
                    "n_benchmarks": len(BENCHMARKS),
                    "n_seeds": len(SEEDS),
                    "greedy_suite_total": float(np.sum(greedy)),
                    "prism_suite_total": float(np.sum(prism)),
                    "suite_total_reduction": (
                        1.0 - float(np.sum(prism)) / float(np.sum(greedy))
                    ),
                    "equal_workload_reduction": float(np.mean(relative)),
                    "wins": wins,
                    "ties": ties,
                    "losses": len(BENCHMARKS) - wins - ties,
                }
            )

        times = {arm: [] for arm in ARMS}
        for benchmark in BENCHMARKS:
            for arm in ARMS:
                group = samples[(benchmark, width, topology, arm, "compile_time_s")]
                if len(group) != len(SEEDS):
                    raise RuntimeError(
                        f"incomplete timing cell: {benchmark}, "
                        f"{width}, {topology}, {arm}"
                    )
                times[arm].append(float(np.mean(group)))
        greedy_time = float(np.mean(times[GREEDY]))
        prism_time = float(np.mean(times[PRISM]))
        timing.append(
            {
                "width": width,
                "topology": topology,
                "n_benchmarks": len(BENCHMARKS),
                "n_seeds": len(SEEDS),
                "greedy_mean_s": greedy_time,
                "prism_mean_s": prism_time,
                "prism_over_greedy": prism_time / greedy_time,
            }
        )

ratios = [row["prism_over_greedy"] for row in timing]
for row in scaling:
    if row["width"] == 32:
        print(
            f"n=32 {row['topology']:<12} {row['metric']:<22}"
            f" {100 * row['suite_total_reduction']:5.1f}%"
            f"  W/T/L {row['wins']}/{row['ties']}/{row['losses']}"
        )
print(
    f"timing ratio mean {np.mean(ratios):.3f},"
    f" range [{min(ratios):.3f}, {max(ratios):.3f}]"
)

### Combined figure

Suite-total reduction curves per topology (top row) and mean
compilation time per arm and topology (bottom), at paper column width.

In [ ]:
scaling_lookup = {
    (int(row["width"]), row["topology"], row["metric"]): 100.0
    * float(row["suite_total_reduction"])
    for row in scaling
}
timing_lookup = {
    (int(row["width"]), row["topology"], arm): float(
        row["greedy_mean_s"] if arm == GREEDY else row["prism_mean_s"]
    )
    for row in timing
    for arm in ARMS
}

with plt.style.context("lusca"):
    plt.rcParams.update(
        {
            "font.size": 7.0,
            "axes.labelsize": 7.2,
            "xtick.labelsize": 6.5,
            "ytick.labelsize": 6.5,
            "legend.fontsize": 6.2,
            "pdf.fonttype": 42,
        }
    )
    figure = plt.figure(figsize=(3.35, 2.55))
    grid_spec = figure.add_gridspec(
        2,
        2,
        height_ratios=(1.0, 1.05),
    )
    axes = (
        figure.add_subplot(grid_spec[0, 0]),
        figure.add_subplot(grid_spec[0, 1]),
        figure.add_subplot(grid_spec[1, :]),
    )

    topology_styles = {
        "grid": (GRID_COLOR, "o", "Square grid"),
        "hex_lattice": (HEX_COLOR, "s", "Hexagonal lattice"),
    }
    reduction_panels = (
        ("router_inserted_swaps", "(a) Routing SWAPs"),
        ("duration_ns", "(b) Critical-path duration"),
    )
    for axis, (metric, title) in zip(
        axes[:2],
        reduction_panels,
        strict=True,
    ):
        for topology in TOPOLOGIES:
            color, marker, label = topology_styles[topology]
            axis.plot(
                WIDTHS,
                [scaling_lookup[(width, topology, metric)] for width in WIDTHS],
                color=color,
                marker=marker,
                markersize=3.2,
                linewidth=1.35,
                label=label,
                zorder=3,
            )
        axis.set_title(title, loc="left", fontsize=7.3)
        axis.set_xticks((8, 16, 24, 32))
        axis.set_ylim(0, 36)
        axis.grid(color="#e1e1e1", linewidth=0.45, zorder=0)
        axis.set_axisbelow(True)
    axes[0].set_ylabel("Reduction (%)")
    axes[0].legend(
        frameon=False,
        loc="lower right",
        handlelength=1.4,
        handletextpad=0.3,
    )

    timing_styles = {
        ("grid", GREEDY): (
            GREEDY_COLOR,
            "o",
            "-",
            "Greedy, grid",
        ),
        ("grid", PRISM): (
            PRISM_COLOR,
            "^",
            "-",
            "PRISM, grid",
        ),
        ("hex_lattice", GREEDY): (
            GREEDY_COLOR,
            "s",
            "--",
            "Greedy, hex",
        ),
        ("hex_lattice", PRISM): (
            PRISM_COLOR,
            "v",
            "--",
            "PRISM, hex",
        ),
    }
    axis = axes[2]
    for topology in TOPOLOGIES:
        for arm in ARMS:
            color, marker, linestyle, label = timing_styles[(topology, arm)]
            axis.plot(
                WIDTHS,
                [timing_lookup[(width, topology, arm)] for width in WIDTHS],
                color=color,
                marker=marker,
                linestyle=linestyle,
                markersize=3.2,
                linewidth=1.35,
                label=label,
                zorder=3,
            )
    axis.set_title("(c) Compilation time", loc="left", fontsize=7.3)
    axis.set_xlabel("Logical qubits")
    axis.set_ylabel("Time (s)")
    axis.set_xticks(WIDTHS)
    axis.set_ylim(bottom=0)
    axis.grid(color="#e1e1e1", linewidth=0.45, zorder=0)
    axis.set_axisbelow(True)
    axis.legend(frameon=False, ncol=2, loc="upper left")

    figure.tight_layout(pad=0.35, w_pad=0.5, h_pad=0.45)

save_figure(figure, "eval_scaling_combined_v24")
plt.show()

## Hardware projection: dual-rail cavity error model

Projects the routed benchmark circuits onto a dual-rail cavity device and
compares the two routing arms (greedy early-commit vs. retained-set) at 8, 12,
and 16 logical qubits on square-grid and hexagonal-lattice topologies.

The model tracks two failure channels per circuit:

- **Photon loss.** Every logical qubit is exposed to cavity decay for the full
  program duration. A terminal loss check detects leakage, so lost shots are
  discarded rather than corrupted; the keep probability
  `exp(-width * duration / T1_cavity)` sets the postselection shot overhead
  `1 / keep`.
- **Residual errors.** Shots that survive the loss check still accumulate
  dephasing and bit-flip errors over the logical-excitation time, plus
  independent residual errors per two-qubit SWS gate and per beamsplitter
  swap. Their product is the conditional fidelity; the figure reports the
  conditional infidelity `1 - fidelity`.

Shorter routed circuits with fewer exchange operations therefore pay less in
both shot overhead and conditional infidelity. The figure reports the
geometric-mean per-workload reduction of both quantities, per topology, as a
function of logical width.

### Parameters

These are the paper's parameter choices, not neutral defaults:

- `scaling16` benchmark population (the 14 hardware benchmarks plus the two
  multiplier circuits) drawn from the width-scaling sweep raw data.
- `cavity_only` loss model: the cavity lifetime applies to all logical-qubit
  excitation time as a state-independent surrogate; state-dependent coupler
  loss is omitted.
- `isa_independent` parameter model: the two characterized transmon roles
  (control, target) are averaged by rate into one representative hardware
  point, residual gate errors are applied independently, and the full
  logical-excitation time accrues dephasing and bit-flip errors.

Hardware constants come from the gate-noise model and Tables 3 and 4 of Liu et
al., *Efficient Routing of Quantum LDPC Codes on Programmable 2D Toric
Architectures* (dual-rail cavity parameters). Times are in microseconds.

In [ ]:
from marabbecca.utils.metrics import device_success

EXPECTED_STARTS = 43
EXPECTED_ITERATIONS = 16
EXPECTED_OBJECTIVE = "swaps"
PARAMETER_MODEL = "isa_independent"

WIDTHS = (8, 12, 16)

# The dual-rail cavity device model (Liu et al., Tables 3 and 4) is
# marabbecca.utils.metrics.device_success.
MODEL_NAMES = (PARAMETER_MODEL,)

### Input data and validation

The raw sweep (read-only) contains one row per (benchmark, width, topology,
seed, arm) routing run; the scaling section above already validated the
full 2240-row population. The
filtered population must be exactly complete, and every row must match the
expected search configuration.

In [ ]:
hw_rows = [
    row
    for row in scale_rows
    if row.get("circuit_name") in BENCHMARKS
    and int(row.get("requested_n", -1)) in WIDTHS
    and row.get("topology_name") in TOPOLOGIES
    and int(row.get("outer_seed", -1)) in SEEDS
    and row.get("strategy") in ARMS
]
expected = {
    (benchmark, width, topology, seed, arm)
    for benchmark in BENCHMARKS
    for width in WIDTHS
    for topology in TOPOLOGIES
    for seed in SEEDS
    for arm in ARMS
}
observed = {
    (
        row["circuit_name"],
        int(row["requested_n"]),
        row["topology_name"],
        int(row["outer_seed"]),
        row["strategy"],
    )
    for row in hw_rows
}
if len(hw_rows) != len(expected) or observed != expected:
    raise RuntimeError("experiment population drift")

for row in hw_rows:
    trace = row.get("search_trace") or {}
    if (
        not row.get("success")
        or row.get("schema_version") != 5
        or row.get("objective") != EXPECTED_OBJECTIVE
        or row.get("heuristic_config") != "qiskit_sabre"
        or row.get("tie_break") != "load"
        or row.get("duration_unmodeled_ops")
        or trace.get("n_starts") != EXPECTED_STARTS
        or trace.get("max_iterations") != EXPECTED_ITERATIONS
        or trace.get("final_swap_trials") != 40
        or trace.get("n_failed_starts") != 0
    ):
        raise RuntimeError(f"invalid result row: {row.get('run_id')}")

print(f"{len(scale_rows)} raw rows, {len(hw_rows)} in population, all valid")

### Per-workload projection

For each routed circuit, the op counts and duration map onto the device model.
Logical `crz` lowers to the two-qubit SWS gate; logical `rx`/`rswap`/`sx` and
routing exchanges lower to beamsplitter swaps; `rz` is software-tracked and
free. With the cavity-only loss model,

    keep = exp(-width * duration / T1_cavity)

and the conditional fidelity multiplies the lifetime factor
`exp(-width * duration * (dephasing_rate + bit_flip_rate))` by
`(1 - eps_SWS)^n_SWS * (1 - eps_1q)^n_rswap`.

Each (benchmark, width, topology, arm) cell averages the five paired seeds.

In [ ]:
def _model_row(row):
    keep_probability, conditional_fidelity = device_success(
        int(row["actual_n_logical"]),
        float(row["duration_ns"]),
        row["program_body_op_counts"],
    )
    return {
        "duration_us": float(row["duration_ns"]) / 1_000.0,
        "keep_probability": keep_probability,
        "postselection_shot_overhead": 1.0 / keep_probability,
        "conditional_fidelity_proxy": conditional_fidelity,
        "conditional_infidelity_proxy": 1.0 - conditional_fidelity,
        "overall_success_probability": keep_probability * conditional_fidelity,
    }


groups = defaultdict(list)
for row in hw_rows:
    groups[
        (
            row["circuit_name"],
            int(row["requested_n"]),
            row["topology_name"],
            row["strategy"],
        )
    ].append(row)

per_workload = []
for key in sorted(groups):
    benchmark, width, topology, arm = key
    group = groups[key]
    if {int(row["outer_seed"]) for row in group} != set(SEEDS):
        raise RuntimeError(f"incomplete seed cell: {key}")
    for model_name in MODEL_NAMES:
        modeled = [_model_row(row) for row in group]
        per_workload.append(
            {
                "benchmark": benchmark,
                "width": width,
                "topology": topology,
                "strategy": arm,
                "model": model_name,
                "n_seeds": len(group),
                **{
                    f"mean_{field}": float(np.mean([item[field] for item in modeled]))
                    for field in modeled[0]
                },
            }
        )

print(f"{len(per_workload)} per-workload rows (seed-averaged); first 12:\n")
header = (
    f"{'benchmark':<24}{'n':>3} {'topology':<12}{'arm':<24}"
    f"{'dur_us':>8}{'keep':>8}{'overhead':>9}{'infid':>9}"
)
print(header)
for item in per_workload[:12]:
    print(
        f"{item['benchmark']:<24}{item['width']:>3} {item['topology']:<12}"
        f"{item['strategy']:<24}{item['mean_duration_us']:>8.2f}"
        f"{item['mean_keep_probability']:>8.3f}"
        f"{item['mean_postselection_shot_overhead']:>9.3f}"
        f"{item['mean_conditional_infidelity_proxy']:>9.4f}"
    )

### Suite summary

For each (width, topology): paired arm comparison over the 16 benchmarks.
Each metric reports the suite-total reduction, the unweighted mean of
workload-relative reductions, and the geometric mean of the workload ratios.
The figure uses the geometric-mean reduction, the natural aggregate for these
positive multiplicative quantities.

In [ ]:
lookup = {
    (
        row["benchmark"],
        int(row["width"]),
        row["topology"],
        row["strategy"],
        row["model"],
    ): row
    for row in per_workload
}
metrics = (
    "duration_us",
    "postselection_shot_overhead",
    "conditional_infidelity_proxy",
)
summary = []
for width in WIDTHS:
    for topology in TOPOLOGIES:
        for model_name in MODEL_NAMES:
            comparison = {
                "width": width,
                "topology": topology,
                "model": model_name,
                "n_benchmarks": len(BENCHMARKS),
                "n_seeds": len(SEEDS),
            }
            for metric in metrics:
                greedy_values = []
                prism_values = []
                relative_reductions = []
                wins = 0
                ties = 0
                for benchmark in BENCHMARKS:
                    greedy = lookup[(benchmark, width, topology, GREEDY, model_name)][
                        f"mean_{metric}"
                    ]
                    prism = lookup[(benchmark, width, topology, PRISM, model_name)][
                        f"mean_{metric}"
                    ]
                    greedy_values.append(greedy)
                    prism_values.append(prism)
                    if greedy == 0:
                        raise RuntimeError(
                            f"zero {metric} denominator: "
                            f"{benchmark}, n={width}, {topology}"
                        )
                    relative_reductions.append(1.0 - prism / greedy)
                    wins += prism < greedy
                    ties += math.isclose(prism, greedy, rel_tol=0.0, abs_tol=1e-12)
                prefix = metric.removesuffix("_us").removesuffix("_proxy")
                comparison |= {
                    f"greedy_mean_{prefix}": float(np.mean(greedy_values)),
                    f"prism_mean_{prefix}": float(np.mean(prism_values)),
                    f"{prefix}_suite_total_reduction": (
                        1.0 - float(np.sum(prism_values)) / float(np.sum(greedy_values))
                    ),
                    f"{prefix}_equal_workload_reduction": float(
                        np.mean(relative_reductions)
                    ),
                    f"{prefix}_geomean_ratio_reduction": (
                        1.0
                        - math.exp(
                            float(np.mean(np.log(prism_values)))
                            - float(np.mean(np.log(greedy_values)))
                        )
                    ),
                    f"{prefix}_wins": wins,
                    f"{prefix}_ties": ties,
                }

            for metric in (
                "keep_probability",
                "conditional_fidelity_proxy",
                "overall_success_probability",
            ):
                prefix = metric.removesuffix("_proxy")
                for arm in ARMS:
                    values = [
                        lookup[(benchmark, width, topology, arm, model_name)][
                            f"mean_{metric}"
                        ]
                        for benchmark in BENCHMARKS
                    ]
                    comparison[f"{arm}_{prefix}_mean"] = float(np.mean(values))

            greedy_success = [
                lookup[(benchmark, width, topology, GREEDY, model_name)][
                    "mean_overall_success_probability"
                ]
                for benchmark in BENCHMARKS
            ]
            prism_success = [
                lookup[(benchmark, width, topology, PRISM, model_name)][
                    "mean_overall_success_probability"
                ]
                for benchmark in BENCHMARKS
            ]
            comparison |= {
                "overall_success_probability_suite_mean_improvement": (
                    float(np.sum(prism_success)) / float(np.sum(greedy_success)) - 1.0
                ),
                "overall_success_probability_equal_workload_improvement": (
                    float(
                        np.mean(
                            [
                                prism / greedy - 1.0
                                for greedy, prism in zip(
                                    greedy_success,
                                    prism_success,
                                    strict=True,
                                )
                            ]
                        )
                    )
                ),
                "overall_success_probability_geomean_ratio_improvement": (
                    math.exp(
                        float(np.mean(np.log(prism_success)))
                        - float(np.mean(np.log(greedy_success)))
                    )
                    - 1.0
                ),
                "overall_success_probability_wins": sum(
                    prism > greedy
                    for greedy, prism in zip(greedy_success, prism_success, strict=True)
                ),
            }
            summary.append(comparison)

print(
    f"{'n':>3} {'topology':<12}"
    f"{'overhead geo-red %':>19}{'infid geo-red %':>17}"
    f"{'dur suite-red %':>17}{'ovhd wins':>10}{'infid wins':>11}"
)
for row in summary:
    print(
        f"{row['width']:>3} {row['topology']:<12}"
        f"{100 * row['postselection_shot_overhead_geomean_ratio_reduction']:>19.1f}"
        f"{100 * row['conditional_infidelity_geomean_ratio_reduction']:>17.1f}"
        f"{100 * row['duration_suite_total_reduction']:>17.1f}"
        f"{row['postselection_shot_overhead_wins']:>7}/16"
        f"{row['conditional_infidelity_wins']:>8}/16"
    )

### Figure

Geometric-mean per-workload reduction from retained-set routing, by logical
width and topology: (a) postselection shot overhead, (b) conditional
infidelity.

In [ ]:
styles = {
    "grid": ("#4c78a8", "o", "Square grid"),
    "hex_lattice": ("#d9902f", "s", "Hexagonal lattice"),
}
panels = (
    (
        "postselection_shot_overhead_geomean_ratio_reduction",
        "(a) Postselection shot overhead",
        "Reduction (%)",
    ),
    (
        "conditional_infidelity_geomean_ratio_reduction",
        "(b) Conditional infidelity",
        "Reduction (%)",
    ),
)
figure_lookup = {
    (int(row["width"]), row["topology"], row["model"]): row for row in summary
}

with plt.style.context("lusca"):
    plt.rcParams.update(
        {
            "font.size": 7.0,
            "axes.labelsize": 7.2,
            "xtick.labelsize": 6.7,
            "ytick.labelsize": 6.7,
            "legend.fontsize": 6.8,
            "pdf.fonttype": 42,
        }
    )
    figure, axes = plt.subplots(1, 2, figsize=(3.35, 1.62), dpi=200)
    for axis, (metric, title, ylabel) in zip(axes, panels, strict=True):
        all_values = []
        for topology in TOPOLOGIES:
            color, marker, label = styles[topology]
            central = np.asarray(
                [
                    100.0 * figure_lookup[(width, topology, PARAMETER_MODEL)][metric]
                    for width in WIDTHS
                ]
            )
            axis.plot(
                WIDTHS,
                central,
                color=color,
                marker=marker,
                markersize=3.2,
                linewidth=1.3,
                label=label,
            )
            axis.annotate(
                f"{central[-1]:.1f}",
                (WIDTHS[-1], central[-1]),
                xytext=(-3, 3),
                textcoords="offset points",
                ha="right",
                fontsize=5.8,
                color=color,
            )
            all_values.extend(central)
        axis.set_title(title, loc="left", fontsize=7.1)
        axis.set_xticks(WIDTHS)
        axis.set_xlabel("Logical qubits")
        axis.set_ylabel(ylabel)
        axis.set_ylim(0, max(all_values) * 1.2)
        axis.grid(color="#e1e1e1", linewidth=0.45)
        axis.set_axisbelow(True)
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 1.02),
    )
    figure.tight_layout(rect=(0, 0, 1, 0.84), pad=0.3, w_pad=0.4)
    save_figure(figure, "eval_hardware_liu_v25")
    plt.show()

### Headline ranges

The two headline ranges quoted in the text must reproduce: the width-16
geometric-mean reductions across the two topologies, for postselection shot
overhead and conditional infidelity.

In [ ]:
EXPECTED_OVERHEAD_RANGE = "28.8-34.2%"
EXPECTED_INFIDELITY_RANGE = "7.8-8.8%"

width_16_rows = [
    row for row in summary if row["width"] == 16 and row["model"] == PARAMETER_MODEL
]
assert len(width_16_rows) == 2
assert {row["topology"] for row in width_16_rows} == set(TOPOLOGIES)

overhead_range = percent_range(
    width_16_rows, "postselection_shot_overhead_geomean_ratio_reduction"
)
infidelity_range = percent_range(
    width_16_rows, "conditional_infidelity_geomean_ratio_reduction"
)
print(f"postselection shot overhead reduction (n=16): {overhead_range}")
print(f"conditional infidelity reduction (n=16): {infidelity_range}")
assert overhead_range == EXPECTED_OVERHEAD_RANGE, overhead_range
assert infidelity_range == EXPECTED_INFIDELITY_RANGE, infidelity_range
print("headline ranges reproduce")